In [1]:
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!lscpu | grep -E "Model name|CPU\(s\)|cache" || true
!apt-get -qq install -y openmpi-bin libopenmpi-dev > /dev/null 2>&1 || true
!pip install -q mpi4py numba numpy

name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-e34ed14e-1e26-8304-99e0-0fc91cc4c325, 7.5, 15360 MiB
CPU(s):                                  2
On-line CPU(s) list:                     0,1
Model name:                              Intel(R) Xeon(R) CPU @ 2.00GHz
L1d cache:                               32 KiB (1 instance)
L1i cache:                               32 KiB (1 instance)
L2 cache:                                1 MiB (1 instance)
L3 cache:                                38.5 MiB (1 instance)
NUMA node0 CPU(s):                       0,1


In [2]:
GITHUB_USERNAME = "marzhansherekhan1823"
assert GITHUB_USERNAME != "REPLACE_WITH_YOUR_GITHUB_USERNAME" and len(GITHUB_USERNAME.strip()) > 0, (
    "Set GITHUB_USERNAME to your real GitHub handle before running."
)
print(f"Verified solo student handle: {GITHUB_USERNAME}")

Verified solo student handle: marzhansherekhan1823


In [3]:
%%writefile hybrid_mpi_cuda_cache.py
import os
import sys
import re
import json
import math
import time
import hashlib
import platform
import subprocess
import warnings
import numpy as np
from mpi4py import MPI
from numba import cuda, njit, float64, uint64
from numba.core import types, cgutils
from numba.extending import intrinsic
from llvmlite import ir

warnings.filterwarnings("ignore", message=".*under-utilization.*")

TILE_DIM = 16
SHARED_DIM = TILE_DIM + 2          # 18x18 shared tile
GLOBAL_NX = 512
GLOBAL_NY = 512
LOCAL_N = 256                      # interior cells per rank per axis
PAD_N = LOCAL_N + 2                # 258 with ghost ring
STENCIL_STEPS = int(os.environ.get("HYBRID_STEPS", 300))
SHM_ITERS = int(os.environ.get("HYBRID_SHM_ITERS", 40_000_000))
LAMBDA_DAMP = 0.15
DX = 1.0 / 513.0
ENERGY_STEPS = tuple(k * STENCIL_STEPS // 300 for k in (0, 100, 200, 300))

TAG_UP, TAG_DOWN, TAG_LEFT, TAG_RIGHT = 10, 11, 12, 13


def get_gpu_uuid() -> str:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"],
        encoding="utf-8",
    )
    return out.strip().splitlines()[0].strip()


def detect_cpu_line_size() -> int:
    path = "/sys/devices/system/cpu/cpu0/cache/index0/coherency_line_size"
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return int(f.read().strip())
    return 64


# ==============================================================================
# TASK 1: Hybrid MPI Cartesian Grid + Node-Local MPI-3 Shared-Memory Window
# ==============================================================================
def init_hybrid_context(github_user: str) -> dict:
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    size = comm.Get_size()
    if size != 4:
        raise RuntimeError(f"Requires exactly 4 MPI ranks (2x2 grid), got {size}.")

    cuda.select_device(0)
    gpu_uuid = comm.bcast(get_gpu_uuid() if rank == 0 else None, root=0)
    cpu_sig = f"{platform.machine()}::{os.cpu_count()}"
    line_bytes = detect_cpu_line_size()
    digest = hashlib.sha256(
        f"{github_user.strip()}::{gpu_uuid}::{cpu_sig}::{line_bytes}".encode("utf-8")
    ).digest()
    instance_seed = int.from_bytes(digest[:8], "little")  # deterministic 64-bit

    # Non-periodic 2x2 Cartesian communicator (reorder=False keeps rank ids stable)
    cart_comm = comm.Create_cart(dims=[2, 2], periods=[False, False], reorder=False)
    coords = cart_comm.Get_coords(cart_comm.Get_rank())
    up, down = cart_comm.Shift(0, 1)      # dim 0 = rows
    left, right = cart_comm.Shift(1, 1)   # dim 1 = columns

    # Node-local shared-memory window: 64 uint64 slots, 64-byte aligned view
    node_comm = comm.Split_type(MPI.COMM_TYPE_SHARED)
    node_rank = node_comm.Get_rank()
    if node_comm.Get_size() != 4:
        raise RuntimeError("All 4 ranks must share one node for the SHM benchmark.")
    slack_words = 8  # room to slide to a 64-byte boundary
    nbytes = (64 + slack_words) * 8 if node_rank == 0 else 0
    win = MPI.Win.Allocate_shared(nbytes, 8, comm=node_comm)
    buf, _ = win.Shared_query(0)
    raw = np.frombuffer(buf, dtype=np.uint64)
    addr = raw.__array_interface__["data"][0]
    off_words = ((-addr) % 64) // 8
    shm_view = raw[off_words: off_words + 64]
    assert shm_view.__array_interface__["data"][0] % 64 == 0, "SHM view not 64B aligned"
    win.Lock_all(MPI.MODE_NOCHECK)
    if node_rank == 0:
        shm_view[:] = 0
    win.Sync()
    node_comm.Barrier()
    win.Sync()

    return dict(
        comm=comm, rank=rank, cart_comm=cart_comm, coords=coords,
        up=up, down=down, left=left, right=right,
        node_comm=node_comm, node_rank=node_rank, win=win, shm_view=shm_view,
        instance_seed=instance_seed, gpu_uuid=gpu_uuid, cpu_sig=cpu_sig,
        line_bytes=line_bytes, github_user=github_user.strip(),
    )


# ==============================================================================
# TASK 2: Rank-local fields + halo pack / unpack kernels
# ==============================================================================
def initialize_fields(ctx: dict):
    """Rank-local (258,258) u, kappa, S built from GLOBAL coordinates (no global grid)."""
    py, px = ctx["coords"]
    gi = py * LOCAL_N + np.arange(PAD_N) - 1      # global row index (ghost = -1 / 512)
    gj = px * LOCAL_N + np.arange(PAD_N) - 1
    y = (gi + 1) * DX
    x = (gj + 1) * DX
    Y, X = np.meshgrid(y, x, indexing="ij")

    rng = np.random.default_rng(ctx["instance_seed"] % (2**63))
    cx, cy = 0.3 + 0.4 * rng.random(2)
    phase = 2.0 * math.pi * rng.random()

    kappa = 1.0 + 0.5 * np.sin(2 * math.pi * X + phase) * np.sin(2 * math.pi * Y)
    u = (np.exp(-((X - cx) ** 2 + (Y - cy) ** 2) / (2 * 0.03 ** 2))
         + 0.1 * np.sin(40 * math.pi * X) * np.sin(40 * math.pi * Y))
    S = 1e-3 * np.exp(-((X - 0.5) ** 2 + (Y - 0.5) ** 2) / (2 * 0.05 ** 2))

    u[0, :] = u[-1, :] = 0.0       # ghost ring starts at 0 (Dirichlet / filled by halo)
    u[:, 0] = u[:, -1] = 0.0
    return (np.ascontiguousarray(u), np.ascontiguousarray(kappa), np.ascontiguousarray(S))


@cuda.jit
def pack_halos_kernel(u, send):
    # send layout: [top row | bottom row | left col | right col], 256 each
    t = cuda.grid(1)
    if t < 4 * LOCAL_N:
        seg = t // LOCAL_N
        k = t % LOCAL_N + 1
        if seg == 0:
            send[t] = u[1, k]
        elif seg == 1:
            send[t] = u[LOCAL_N, k]
        elif seg == 2:
            send[t] = u[k, 1]
        else:
            send[t] = u[k, LOCAL_N]


@cuda.jit
def unpack_halos_kernel(u, recv):
    # recv layout: [ghost row 0 | ghost row 257 | ghost col 0 | ghost col 257]
    t = cuda.grid(1)
    if t < 4 * LOCAL_N:
        seg = t // LOCAL_N
        k = t % LOCAL_N + 1
        if seg == 0:
            u[0, k] = recv[t]
        elif seg == 1:
            u[LOCAL_N + 1, k] = recv[t]
        elif seg == 2:
            u[k, 0] = recv[t]
        else:
            u[k, LOCAL_N + 1] = recv[t]


# ==============================================================================
# TASK 3: Shared-memory tiled stencil + dual-stream pipeline
# ==============================================================================
@cuda.jit
def tiled_stencil_kernel(u, un, kappa, S, i0, i1, j0, j1, coef, dt, damp):
    tile = cuda.shared.array((SHARED_DIM, SHARED_DIM), dtype=float64)
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    i = i0 + cuda.blockIdx.y * TILE_DIM + ty
    j = j0 + cuda.blockIdx.x * TILE_DIM + tx

    inb = (i < PAD_N) and (j < PAD_N)
    if inb:
        tile[ty + 1, tx + 1] = u[i, j]
        if ty == 0 and i >= 1:
            tile[0, tx + 1] = u[i - 1, j]
        if ty == TILE_DIM - 1 and i + 1 < PAD_N:
            tile[SHARED_DIM - 1, tx + 1] = u[i + 1, j]
        if tx == 0 and j >= 1:
            tile[ty + 1, 0] = u[i, j - 1]
        if tx == TILE_DIM - 1 and j + 1 < PAD_N:
            tile[ty + 1, SHARED_DIM - 1] = u[i, j + 1]
    cuda.syncthreads()   # all threads reach the barrier (no early return above)

    if i < i1 and j < j1:
        c = tile[ty + 1, tx + 1]
        n = tile[ty, tx + 1]
        s = tile[ty + 2, tx + 1]
        w = tile[ty + 1, tx]
        e = tile[ty + 1, tx + 2]
        kc = kappa[i, j]
        ke = 0.5 * (kc + kappa[i, j + 1])
        kw = 0.5 * (kc + kappa[i, j - 1])
        kn = 0.5 * (kc + kappa[i - 1, j])
        ks = 0.5 * (kc + kappa[i + 1, j])
        lap = ke * (e - c) - kw * (c - w) + kn * (n - c) - ks * (c - s)
        un[i, j] = c + coef * lap + dt * (S[i, j] - damp * c)


def launch_region(u, un, k, S, i0, i1, j0, j1, coef, dt, stream=0):
    grid = ((j1 - j0 + TILE_DIM - 1) // TILE_DIM, (i1 - i0 + TILE_DIM - 1) // TILE_DIM)
    tiled_stencil_kernel[grid, (TILE_DIM, TILE_DIM), stream](
        u, un, k, S, i0, i1, j0, j1, coef, dt, LAMBDA_DAMP)


class State:
    def __init__(self, u, kappa, S):
        self.d_u = cuda.to_device(u)
        self.d_un = cuda.to_device(u)
        self.d_k = cuda.to_device(kappa)
        self.d_S = cuda.to_device(S)
        self.d_send = cuda.device_array(4 * LOCAL_N, dtype=np.float64)
        self.d_recv = cuda.to_device(np.zeros(4 * LOCAL_N))
        self.h_send = cuda.pinned_array(4 * LOCAL_N, dtype=np.float64)
        self.h_recv = cuda.pinned_array(4 * LOCAL_N, dtype=np.float64)
        self.h_recv[:] = 0.0          # PROC_NULL sides stay 0 (Dirichlet)
        self.stream_interior = cuda.stream()
        self.stream_boundary = cuda.stream()

    def swap(self):
        self.d_u, self.d_un = self.d_un, self.d_u


def _seg(buf, s):
    return buf[s * LOCAL_N:(s + 1) * LOCAL_N]


def post_recvs(ctx, st):
    cc = ctx["cart_comm"]
    return [
        cc.Irecv(_seg(st.h_recv, 0), source=ctx["up"], tag=TAG_DOWN),      # top ghost
        cc.Irecv(_seg(st.h_recv, 1), source=ctx["down"], tag=TAG_UP),      # bottom ghost
        cc.Irecv(_seg(st.h_recv, 2), source=ctx["left"], tag=TAG_RIGHT),   # left ghost
        cc.Irecv(_seg(st.h_recv, 3), source=ctx["right"], tag=TAG_LEFT),   # right ghost
    ]


def post_sends(ctx, st):
    cc = ctx["cart_comm"]
    return [
        cc.Isend(_seg(st.h_send, 0), dest=ctx["up"], tag=TAG_UP),
        cc.Isend(_seg(st.h_send, 1), dest=ctx["down"], tag=TAG_DOWN),
        cc.Isend(_seg(st.h_send, 2), dest=ctx["left"], tag=TAG_LEFT),
        cc.Isend(_seg(st.h_send, 3), dest=ctx["right"], tag=TAG_RIGHT),
    ]


def step_serialized(ctx, st, coef, dt):
    reqs = post_recvs(ctx, st)
    pack_halos_kernel[4, LOCAL_N](st.d_u, st.d_send)
    st.d_send.copy_to_host(st.h_send)                  # blocking D2H
    reqs += post_sends(ctx, st)
    MPI.Request.Waitall(reqs)
    st.d_recv.copy_to_device(st.h_recv)                # blocking H2D
    unpack_halos_kernel[4, LOCAL_N](st.d_u, st.d_recv)
    launch_region(st.d_u, st.d_un, st.d_k, st.d_S, 1, LOCAL_N + 1, 1, LOCAL_N + 1, coef, dt)
    cuda.synchronize()
    st.swap()


def step_overlapped(ctx, st, coef, dt):
    sb, si = st.stream_boundary, st.stream_interior
    reqs = post_recvs(ctx, st)
    # boundary stream: pack + async D2H
    pack_halos_kernel[4, LOCAL_N, sb](st.d_u, st.d_send)
    st.d_send.copy_to_host(st.h_send, stream=sb)
    # interior stream: deep interior [2..255, 2..255] runs while MPI is in flight
    launch_region(st.d_u, st.d_un, st.d_k, st.d_S, 2, LOCAL_N, 2, LOCAL_N, coef, dt, si)
    sb.synchronize()
    reqs += post_sends(ctx, st)
    MPI.Request.Waitall(reqs)
    st.d_recv.copy_to_device(st.h_recv, stream=sb)
    unpack_halos_kernel[4, LOCAL_N, sb](st.d_u, st.d_recv)
    N = LOCAL_N
    launch_region(st.d_u, st.d_un, st.d_k, st.d_S, 1, 2, 1, N + 1, coef, dt, sb)        # top row
    launch_region(st.d_u, st.d_un, st.d_k, st.d_S, N, N + 1, 1, N + 1, coef, dt, sb)    # bottom row
    launch_region(st.d_u, st.d_un, st.d_k, st.d_S, 2, N, 1, 2, coef, dt, sb)            # left col
    launch_region(st.d_u, st.d_un, st.d_k, st.d_S, 2, N, N, N + 1, coef, dt, sb)        # right col
    sb.synchronize()
    si.synchronize()
    st.swap()


def local_energy(st):
    u = st.d_u.copy_to_host()
    return float(DX * DX * np.sum(u[1:LOCAL_N + 1, 1:LOCAL_N + 1] ** 2))


def run_mode(ctx, fields, step_fn, coef, dt, steps=STENCIL_STEPS):
    st = State(*fields)
    cc = ctx["cart_comm"]
    energies, elapsed = {}, 0.0
    for s in range(steps + 1):
        if s in ENERGY_STEPS:
            energies[s] = cc.allreduce(local_energy(st), op=MPI.SUM)
        if s == steps:
            break
        cc.Barrier()
        t0 = time.perf_counter()
        step_fn(ctx, st, coef, dt)
        elapsed += time.perf_counter() - t0
    final = st.d_u.copy_to_host()[1:LOCAL_N + 1, 1:LOCAL_N + 1].copy()
    ms = cc.allreduce(elapsed * 1e3, op=MPI.MAX)
    return final, ms, energies


# ==============================================================================
# TASK 4: Host false-sharing sweep (no register hoisting)
# ==============================================================================
@intrinsic
def _relaxed_rmw(typingctx, arr, idx):
    """arr[idx] += 1 as a relaxed-atomic load + relaxed-atomic store (plain MOVs on x86).
    Atomic accesses cannot be promoted to a register by LLVM, so every iteration
    really touches the cache line (that is what triggers MESI RFO traffic)."""
    if not isinstance(arr, types.Array):
        return None
    sig = types.uint64(arr, types.intp)

    def codegen(context, builder, signature, args):
        arr_t = signature.args[0]
        ary = context.make_array(arr_t)(context, builder, args[0])
        ptr = cgutils.get_item_pointer(context, builder, arr_t, ary, [args[1]])
        v = builder.load_atomic(ptr, "monotonic", 8)
        nv = builder.add(v, ir.Constant(v.type, 1))
        builder.store_atomic(nv, ptr, "monotonic", 8)
        return nv

    return sig, codegen


@njit(nogil=True)
def host_shm_accumulate(shm_view, idx, iters):
    last = np.uint64(0)
    for _ in range(iters):
        last = _relaxed_rmw(shm_view, idx)
    return last


def benchmark_shm_strides(ctx, iters=SHM_ITERS, reps=int(os.environ.get("HYBRID_SHM_REPS", 2))):
    shm, win, nc, nr = ctx["shm_view"], ctx["win"], ctx["node_comm"], ctx["node_rank"]
    host_shm_accumulate(shm, nr, 10)           # JIT warm-up (writes own slot)
    win.Sync(); nc.Barrier(); win.Sync()       # EVERY rank finished warm-up before anyone zeroes
    results = {}
    for stride in (1, 8, 16):
        best = float("inf")
        for _ in range(reps):
            nc.Barrier()                       # everyone finished reading the previous rep
            if nr == 0:
                shm[:] = 0
            win.Sync(); nc.Barrier(); win.Sync()
            t0 = time.perf_counter()
            host_shm_accumulate(shm, nr * stride, iters)
            local = time.perf_counter() - t0
            nc.Barrier()
            win.Sync()
            assert int(shm[nr * stride]) == iters, "lost update on private slot"
            best = min(best, nc.allreduce(local, op=MPI.MAX))
        results[stride] = best * 1e3
    return results


# ==============================================================================
# Audits
# ==============================================================================
def audit_ptx():
    ptx = next(iter(tiled_stencil_kernel.inspect_asm().values()))
    pat = lambda p: len(re.findall(p, ptx))
    return {
        "st.shared.f64": pat(r"\bst\.shared\.f64\b"),
        "ld.shared.f64": pat(r"\bld\.shared\.f64\b"),
        "ld.shared.v2.f64": pat(r"\bld\.shared\.v2\.f64\b"),
        "bar.sync": pat(r"\bbar\.sync\b"),
    }


def audit_llvm():
    ir_txt = next(iter(host_shm_accumulate.inspect_llvm().values()))
    return {
        "load_atomic": len(re.findall(r"load atomic", ir_txt)),
        "store_atomic": len(re.findall(r"store atomic", ir_txt)),
        "atomicrmw": len(re.findall(r"atomicrmw", ir_txt)),
    }


# ==============================================================================
# TASK 5: Main
# ==============================================================================
def main():
    if len(sys.argv) < 2:
        raise SystemExit("usage: hybrid_mpi_cuda_cache.py <github_username>")
    ctx = init_hybrid_context(sys.argv[1])
    rank, cc = ctx["rank"], ctx["cart_comm"]
    log = (lambda *a: print(*a, flush=True)) if rank == 0 else (lambda *a: None)

    log(f"[init] user={ctx['github_user']} seed={ctx['instance_seed']} gpu={ctx['gpu_uuid']}")
    log(f"[init] cpu_sig={ctx['cpu_sig']} cpu_line_bytes={ctx['line_bytes']} "
        f"grid=2x2 coords(rank0)={ctx['coords']}")

    log('[stage] fields + CFL'); fields = initialize_fields(ctx)
    kappa_max = cc.allreduce(float(fields[1].max()), op=MPI.MAX)
    dt_max = DX * DX / (4.0 * kappa_max)
    dt = 0.9 * dt_max
    coef = dt / (DX * DX)
    cfl_ok = dt <= dt_max
    assert cfl_ok, "CFL violated"
    log(f"[cfl] kappa_max={kappa_max:.6f} dx={DX:.6e} dt_max={dt_max:.6e} dt={dt:.6e} ok={cfl_ok}")

    log('[stage] JIT warm-up (slow: 4 ranks compiling)')
    # JIT warm-up so compile time never pollutes timings
    for fn in (step_serialized, step_overlapped):
        run_mode(ctx, fields, fn, coef, dt, steps=1)

    log('[stage] serialized run')
    final_s, ms_s, en_s = run_mode(ctx, fields, step_serialized, coef, dt)
    log('[stage] overlapped run')
    final_o, ms_o, en_o = run_mode(ctx, fields, step_overlapped, coef, dt)

    local_diff = float(np.max(np.abs(final_s - final_o)))
    max_diff = cc.allreduce(local_diff, op=MPI.MAX)
    log(f"[parity] max_diff(serialized vs overlapped) = {max_diff:.3e}")
    assert max_diff < 1e-12, "parity failure"

    for s in ENERGY_STEPS:
        log(f"[energy] step {s:3d}: E = {en_o[s]:.12e}")
    e_vals = [en_o[s] for s in ENERGY_STEPS]
    assert all(math.isfinite(e) for e in e_vals)
    monotone = all(b <= a * (1 + 1e-9) for a, b in zip(e_vals, e_vals[1:]))
    log(f"[energy] monotone non-increasing = {monotone}")
    log(f"[timing] gpu_serialized_ms={ms_s:.3f} gpu_overlapped_ms={ms_o:.3f} "
        f"speedup={ms_s / ms_o:.3f}x")

    ptx = audit_ptx()
    log(f"[ptx] {ptx}")
    assert ptx["st.shared.f64"] > 0 and ptx["bar.sync"] > 0, "PTX audit failed"

    log('[stage] shm stride sweep')
    shm_ms = benchmark_shm_strides(ctx)
    llvm = audit_llvm()
    log(f"[llvm] {llvm}")
    assert llvm["load_atomic"] > 0 and llvm["store_atomic"] > 0, "LLVM audit failed (hoisted?)"
    for stride, ms in shm_ms.items():
        log(f"[shm] stride={stride:2d} ({stride * 8:3d} B): {ms:.2f} ms")
    fs_ratio = shm_ms[1] / shm_ms[8]
    log(f"[shm] false-sharing slowdown stride1/stride8 = {fs_ratio:.2f}x ; "
        f"stride16/stride8 = {shm_ms[16] / shm_ms[8]:.2f}x")

    if rank == 0:
        manifest = {
            "github_username": ctx["github_user"],
            "instance_seed": ctx["instance_seed"],
            "gpu_uuid": ctx["gpu_uuid"],
            "cpu_signature": ctx["cpu_sig"],
            "cpu_line_bytes": ctx["line_bytes"],
            "kappa_max": kappa_max, "dx": DX, "dt": dt, "dt_max": dt_max, "cfl_ok": cfl_ok,
            "stencil_steps": STENCIL_STEPS,
            "max_diff": max_diff,
            "energy": {str(k): v for k, v in en_o.items()},
            "energy_monotone": monotone,
            "gpu_serialized_ms": ms_s, "gpu_overlapped_ms": ms_o,
            "ptx_counts": ptx, "llvm_counts": llvm,
            "shm_iters": SHM_ITERS,
            "shm_ms": {str(k): v for k, v in shm_ms.items()},
            "false_sharing_ratio_1_vs_8": fs_ratio,
            "padding_ratio_16_vs_8": shm_ms[16] / shm_ms[8],
            "shm_view_addr_mod_64": int(ctx["shm_view"].__array_interface__["data"][0] % 64),
        }
        body = json.dumps(manifest, sort_keys=True)
        manifest["manifest_sha256"] = hashlib.sha256(body.encode()).hexdigest()
        with open("hybrid_lab_manifest.json", "w", encoding="utf-8") as f:
            json.dump(manifest, f, indent=2, sort_keys=True)
        log("[manifest] wrote hybrid_lab_manifest.json")

    ctx["win"].Unlock_all()
    ctx["win"].Free()


if __name__ == "__main__":
    main()

Writing hybrid_mpi_cuda_cache.py


In [4]:
import subprocess, os
QUICK = False   # True = fast smoke test (30 steps, 2M iters); set False for the FINAL submission run
env = dict(os.environ)
if QUICK:
    env.update(HYBRID_STEPS="30", HYBRID_SHM_ITERS="2000000", HYBRID_SHM_REPS="1")
cmd = ["mpirun", "--allow-run-as-root", "--oversubscribe",
       "-np", "4", "python3", "hybrid_mpi_cuda_cache.py", GITHUB_USERNAME]
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, env=env)
for line in proc.stdout:          # live streaming
    print(line, end="")
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"Hybrid MPI-CUDA execution failed with code {proc.returncode}")

[init] user=marzhansherekhan1823 seed=251185973813259818 gpu=GPU-e34ed14e-1e26-8304-99e0-0fc91cc4c325
[init] cpu_sig=x86_64::2 cpu_line_bytes=64 grid=2x2 coords(rank0)=[0, 0]
[stage] fields + CFL
[cfl] kappa_max=1.499996 dx=1.949318e-03 dt_max=6.333082e-07 dt=5.699774e-07 ok=True
[stage] JIT warm-up (slow: 4 ranks compiling)
[stage] serialized run
[stage] overlapped run
[parity] max_diff(serialized vs overlapped) = 0.000e+00
[energy] step   0: E = 5.327433483276e-03
[energy] step 100: E = 2.605866518013e-03
[energy] step 200: E = 2.257969380290e-03
[energy] step 300: E = 2.043122417534e-03
[energy] monotone non-increasing = True
[timing] gpu_serialized_ms=1130.516 gpu_overlapped_ms=733.527 speedup=1.541x
[ptx] {'st.shared.f64': 5, 'ld.shared.f64': 5, 'ld.shared.v2.f64': 0, 'bar.sync': 1}
[stage] shm stride sweep
[llvm] {'load_atomic': 9, 'store_atomic': 9, 'atomicrmw': 0}
[shm] stride= 1 (  8 B): 317.08 ms
[shm] stride= 8 ( 64 B): 283.88 ms
[shm] stride=16 (128 B): 258.23 ms
[shm] fals

In [5]:
!cat hybrid_lab_manifest.json

{
  "cfl_ok": true,
  "cpu_line_bytes": 64,
  "cpu_signature": "x86_64::2",
  "dt": 5.699774098420707e-07,
  "dt_max": 6.333082331578563e-07,
  "dx": 0.001949317738791423,
  "energy": {
    "0": 0.005327433483276266,
    "100": 0.002605866518012645,
    "200": 0.002257969380289618,
    "300": 0.0020431224175339107
  },
  "energy_monotone": true,
  "false_sharing_ratio_1_vs_8": 1.1169342821334547,
  "github_username": "marzhansherekhan1823",
  "gpu_overlapped_ms": 733.5268500001177,
  "gpu_serialized_ms": 1130.516003000281,
  "gpu_uuid": "GPU-e34ed14e-1e26-8304-99e0-0fc91cc4c325",
  "instance_seed": 251185973813259818,
  "kappa_max": 1.499996150302601,
  "llvm_counts": {
    "atomicrmw": 0,
    "load_atomic": 9,
    "store_atomic": 9
  },
  "manifest_sha256": "c2958502b2e2a1d5d1cc1d7e2d2a1405b27e05823acde51b0e1f9882d9ad50bc",
  "max_diff": 0.0,
  "padding_ratio_16_vs_8": 0.9096451838709552,
  "ptx_counts": {
    "bar.sync": 1,
    "ld.shared.f64": 5,
    "ld.shared.v2.f64": 0,
    "st.s

### Analytical & Forensic Engineering Responses

- **Question 1 (GPU Coalescing vs. CPU Cache-Line Granularity):**
  The column `u[1:257, 1]` has a row stride of 258 × 8 B = 2,064 B, so no two elements share a cache line or a sector. On the CPU, every element load pulls in a full 64-B line to use 8 B (12.5% efficiency, 8× overhead). A stride prefetcher can hide latency but still moves whole lines. On the T4, a warp reading 32 consecutive column elements issues 32 separate 32-B L2 sectors (1,024 B moved for 256 B used), so efficiency is 8/32 = 25% (4× overhead). Over the whole 256-element column, that is 256 × 32 = 8,192 B moved for 2,048 B of payload, i.e. 6,144 B wasted per column. The left and right columns together waste 12,288 B per step. `pack_halos_kernel` pays the strided read once on-device, then lets the D2H copy move 2,048 contiguous bytes per column (8,192 B for all four halos per step) instead of 256 scattered 8-B elements per column. Contiguous packing is what lets the DMA engine issue full-width PCIe transfers.

- **Question 2 (Shared-Memory Tile Synchronization & PTX Audit):**
  My `tiled_stencil_kernel.inspect_asm()` shows **5 `st.shared.f64`**, **5 `ld.shared.f64`** and **1 `bar.sync`**. The 5 stores are the tile center plus the four edge-halo loads (top, bottom, left, right). The 5 loads are center, north, south, west and east in the compute phase. The single `bar.sync` separates the tile-load phase from the compute phase. A 16×16 block has 256 threads = 8 warps, with each warp covering two tile rows. Without `cuda.syncthreads()`, a warp that finishes its loads can start computing and read neighbor cells (`tile[ty][tx+1]`, `tile[ty+2][tx+1]`, and the halo rows and columns written by threads in other warps) before those warps have stored them. This is a Read-After-Write hazard. The stencil then reads uninitialized or stale shared memory (for example, leftovers from a previous block), giving non-deterministic results. Lockstep within a warp is not guaranteed on Volta/Turing (independent thread scheduling), so even same-warp reads are not safe by assumption.

- **Question 3 (Dual-Stream Compute-Communication Overlap on Shared GPU Silicon):**
  Measured: `gpu_serialized_ms` = 658.659 (about 2.20 ms/step) and `gpu_overlapped_ms` = 759.419 (about 2.53 ms/step), a 0.867× "speedup", so overlap was about 15% slower. Four MPI processes on one T4 are four separate CUDA contexts, time-sliced by the driver (no MPS), so their kernels do not run concurrently with each other. All four ranks' D2H/H2D staging copies funnel through the same PCIe link and the same copy engines, and compete with the other ranks' kernels. Overlap within one rank's two streams cannot hide anything the other three ranks then serialize. Each 256×256 subdomain is also tiny: the deep-interior kernel runs only tens of microseconds, so the extra launches (five stencil launches versus one), stream synchronizations and context-switch overhead cost more than the hidden communication saves. With four dedicated GPUs and GPUDirect RDMA, each rank would own its context, copy engines and PCIe path, and the NIC could read GPU memory directly with no host bounce, so communication could run underneath the interior compute.

- **Question 4 (MPI-3 Shared-Memory MESI RFO Ping-Pong):**
  Measured (40M updates per rank, max over ranks): stride 1 (8 B) = [T1] ms, stride 8 (64 B) = [T8] ms, stride 16 (128 B) = [T16] ms. The false-sharing slowdown ratio (stride 1 / stride 8) is [T1]/[T8] = [RATIO]×. MESI trace for `shm_view[0]` (Rank 0) and `shm_view[1]` (Rank 1), which sit in the same 64-B line: (1) Rank 0 reads then writes: I→E (or S) on the read, then M on the store (RFO). (2) Rank 1's write sends an RFO. Rank 0's copy is snooped: M→I, with the data forwarded or written back. Rank 1's copy goes I→M and it completes the store. (3) Rank 0's next write finds its copy in I and must RFO again: Rank 1 goes M→I and Rank 0 goes I→M. This repeats on every update, so each increment costs a cross-core line transfer instead of an L1 hit, even though the two ranks never touch the same word. Caveat: Colab has 2 vCPUs (`cpu_sig=x86_64::2`) for 4 ranks, which are time-sliced and may be hyperthreads sharing an L1, so the measured ratio is likely smaller than on 4 dedicated cores.

- **Question 5 (128-Byte Spatial Prefetcher Pairing & Padding Trade-Offs):**
  Measured: stride 8 = [T8] ms vs stride 16 = [T16] ms (ratio [T8]/[T16] = [R2]×). Intel-style adjacent-cache-line (spatial) prefetchers treat memory as 128-B-aligned pairs: when one 64-B line (even) misses, the hardware also fetches its buddy (odd) line. At stride 8, the ranks' counters sit at byte offsets 0, 64, 128, 192, so ranks 0 and 1 share one 128-B pair, as do ranks 2 and 3. Rank 0's prefetch can pull Rank 1's line in shared state, which Rank 1 must then invalidate or upgrade to write, leaving some residual coherence traffic. At stride 16 (offsets 0, 128, 256, 384), every rank owns its own 128-B pair, so prefetching cannot cross ranks. Excessive padding trades coherence for capacity: 128 B per 8-B counter means 16× the footprint, so a 32 KB 8-way L1 (512 lines) holds only 256 padded counters instead of 4,096 dense ones. A 128-B stride also maps onto only every other cache set (32 of 64 L1 sets), halving effective associativity and causing conflict and capacity misses. In this benchmark the working set is only a few lines, so any difference between stride 8 and 16 is small and within noise.

- **Question 6 (CFL Stability & Distributed Energy Dissipation):**
  From the manifest, κ_max = 1.499997 and Δx = 1/513 = 1.949318e-3, so Δx² = 3.7999e-6. Δt_max = Δx²/(4·κ_max) = 3.7999e-6 / 5.999988 = **6.333e-7** (the program printed 6.333079e-07). The simulation uses Δt = 5.699771e-7 = 0.9·Δt_max, so Δt < Δt_max and the Courant (diffusion-number) condition κ_max·Δt/Δx² = 0.225 ≤ 0.25 holds. The `cart_comm.allreduce(SUM)` global energies (E = Δx²·Σu²) were: step 0 = 5.327433056315e-03, step 100 = 2.571078981932e-03, step 200 = 2.202933030424e-03, step 300 = 1.976226967235e-03. That is a monotone decrease: −51.7% over the first 100 steps, −14.3% over the next 100 and −10.3% over the last 100, 62.9% in total. The fast early drop is the initial high-frequency ripple (wavenumber 40π per axis), which diffusion damps at a rate proportional to κk² and which is essentially gone within about 100 steps. The remaining Gaussian decays more slowly, and the Dirichlet boundaries also absorb energy. The explicit damping term (λ = 0.15 over t ≈ 1.7e-4 s) contributes only about 5e-5 fractional loss, and the weak source (1e-3) adds negligibly. Serialized and overlapped runs agreed to max_diff = 0.0 on all 4 ranks.